# Many distributions at once

**Worksheet · Data 110 · Fall 2026 · Week 6**
Rebin Muhammad

This is the worksheet for the lecture. Do it in class, after the slides. Run each cell with **Shift+Enter**, from the top.

Under each exercise there is an empty cell. Write your solution in that cell, then run it. This file does not contain the answers.

## How to open this in Google Colab

1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this file.
3. Upload `lincoln_temps.csv` before you run the data cell. Click the **folder icon** on the left, then the **upload icon**, and choose the file. The name must stay `lincoln_temps.csv`. Colab forgets uploaded files when the session ends, so you upload it again next time.


## Setup

This cell loads the packages. `KernelDensity` is the same tool as last week: it turns a list of temperatures into a density curve.


In [ ]:
import matplotlib.pyplot as plt  # pyplot draws the plots. We call it plt.
import numpy as np               # numpy does the math. We call it np.
import pandas as pd              # pandas holds tables. We call it pd.
from sklearn.neighbors import KernelDensity  # KernelDensity computes a density curve.

# rcParams is matplotlib's list of default settings.
# A change here applies to every plot after this cell.
# The spines are the four lines that make the frame.
# False hides the top line and the right line.
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

FILL = "#56B4E9"  # the blue used for a single series in the lecture


## The temperature table

One row is one day in Lincoln, Nebraska, in 2016.

- `date` is the calendar day.
- `month` is a three-letter name, `Jan` through `Dec`.
- `month_long` is the full name, `January` through `December`.
- `mean_temp` is that day's average temperature, in degrees Fahrenheit.

`read_csv` turns the file into a table. `head()` shows the first five rows.


In [ ]:
temps = pd.read_csv("lincoln_temps.csv")

# months is the order of the year. A plot that sorted the names
# would put April before August, which is not the order of the year.
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

print(len(temps), "days")
temps.head()


## One list of temperatures per month

`loc` keeps the rows that pass a test. This cell builds twelve lists, one per month, in the order of `months`. Later cells draw those lists.


In [ ]:
groups = [
    temps.loc[temps["month"] == month, "mean_temp"]
    for month in months
]
[len(group) for group in groups]


## 1. Boxplots

**Exercise.** Draw a boxplot of `groups`. Put the month names on the horizontal axis, and label the vertical axis `mean temperature (°F)`.

`boxplot` places the first list at x = 1, the second at x = 2, and so on. `set_xticks(range(1, 13))` marks those twelve positions. `set_xticklabels(months)` writes the names.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## 2. Violins

**Exercise.** Draw the same twelve lists as violins. Pass `showmedians=True` so each violin has a line at the median. Use the same month names and the same vertical label.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## 3. A strip chart

**Exercise.** Draw every day as a dot, with no sideways nudge. Use `enumerate(months, start=1)` so January sits at x = 1.

`np.full(len(y), i)` makes a list of the number `i`, once per day in that month. Every day of a month then shares one x position.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## 4. Nudge the dots sideways

**Exercise.** Repeat the strip chart, but add a horizontal shift of `rng.uniform(-0.15, 0.15, size=len(y))` to the x position.

`default_rng(320)` starts the random numbers at a fixed place, so the picture does not change every time you run the cell. The shift is in month-units, not in degrees. A day's temperature stays what it was.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## 5. What a vertical nudge does

**Exercise.** Copy the jittered strip chart. After you have `y`, add a second shift: `y = y + rng.uniform(-2, 2, size=len(y))`.

Look at the picture, then answer in a comment at the top of the cell: a vertical nudge changes the temperature the dot claims. That is why the lecture keeps the vertical nudge at zero.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## 6. Sina plots

A sina plot is a strip chart whose sideways nudge follows the violin. Where a temperature is common, the violin is wide and the dots may spread. Where it is rare, they stay near the center.

`sina_x` returns an x position for each day. `bandwidth=3.4` is the width of each day's bell, in degrees, the same choice as the lecture figures. `score_samples` returns the log of the density, and `np.exp` turns that back into the density. Dividing by `height.max()` makes the most common temperature have width 1, before we multiply by `width`.


In [ ]:
def sina_x(values, center, width=0.36, seed=320):
    """Place each day inside the violin, wider where that temperature is common."""
    values = np.asarray(values, dtype=float).reshape(-1, 1)
    # bandwidth is the bell's standard deviation, in degrees Fahrenheit.
    model = KernelDensity(kernel="gaussian", bandwidth=3.4)
    model.fit(values)
    height = np.exp(model.score_samples(values))
    height = height / height.max()
    rng = np.random.default_rng(seed)
    shift = (rng.random(len(values)) * 2 - 1) * height * width
    return center + shift


**Exercise.** Draw the violins with `showextrema=False` and no median line, fill them with `#D6EAF8`, and remove the outline. Then scatter the days at `sina_x(y, i)`.

`drawn["bodies"]` is the list of violin shapes. `set_edgecolor("none")` removes the outline.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## 7. Ridgelines

A ridgeline is one density per month, each lifted onto its own row. January is the top row. The height of each curve is divided by the tallest curve in the year, so the rows share a scale.

`bandwidth` is still the width of the bell, in degrees. A small bandwidth follows a few unusual days. A large one washes the month into one lump.


In [ ]:
def ridges(frame, bandwidth=3.4):
    """One density per month. January is the top row."""
    grid = np.linspace(-20, 105, 400).reshape(-1, 1)
    fig, ax = plt.subplots(figsize=(8, 5.2))
    names = ["January", "February", "March", "April", "May", "June",
             "July", "August", "September", "October", "November", "December"]
    curves = []
    for name in names:
        values = frame.loc[frame["month_long"] == name, "mean_temp"].to_numpy()
        model = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
        model.fit(values.reshape(-1, 1))
        curve = np.exp(model.score_samples(grid))
        curves.append(curve)
    peak = max(curve.max() for curve in curves)
    for i, curve in enumerate(curves):
        baseline = 11 - i
        height = curve / peak * 2.4
        ax.fill_between(grid.ravel(), baseline, baseline + height, color=FILL, alpha=0.85)
    ax.set_yticks(range(12))
    ax.set_yticklabels(list(reversed(names)))
    ax.set_xlabel("mean temperature (°F)")
    ax.set_xlim(-15, 100)
    return ax


**Exercise.** Call `ridges(temps)` once. Then call it again with `bandwidth=1`.

The second picture should look bumpier. Those bumps are a few days, and also noise.

### Your solution

Write your code in the next cell, then run it.


In [ ]:
# Write your solution here.


## What to keep

- **Boxplot.** The box is the middle half of the days. The line is the median. A dot is a day beyond the fence.
- **Violin.** A density on its side, mirrored. The widest point is the most common temperature.
- **Strip chart.** One dot per day. Nudge sideways so equal temperatures do not stack. A vertical nudge changes the temperature.
- **Ridgeline.** One density per month, each on its own row.

The argument follows Claus O. Wilke, *Fundamentals of Data Visualization*, Chapter 7. This worksheet is the Python version for Data 110, Fall 2026.
